In [1]:
# import os
# import pandas as pd
# from sklearn.model_selection import train_test_split
# import tensorflow as tf
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.applications import ResNet50
# from tensorflow.keras.layers import Conv2D, Reshape, UpSampling2D, Flatten, Input, LayerNormalization, MultiHeadAttention, Dense
# from tensorflow.keras.models import Model

# # Set paths
# data_dir = "/kaggle/input/isic-2020-jpg-224x224-resized/train-image/image"  # Path to the folder containing all images
# csv_path = "/kaggle/input/isic-2020-jpg-224x224-resized/train-metadata.csv"  # Path to the CSV file

# # Load the dataset
# labels_df = pd.read_csv(csv_path)

# # Add extensions to image filenames (assuming image names are in 'isic_id' column)
# labels_df['image'] = labels_df['isic_id'] + '.jpg'

# # Define class column (binary classification)
# class_column = 'target'  # Assuming the column for binary labels is 'target'

# # Split the dataset into train, validation, and test sets
# train_df, test_val_df = train_test_split(labels_df, test_size=0.2, random_state=42)
# val_df, test_df = train_test_split(test_val_df, test_size=0.5, random_state=42)

# train_df['target'] = train_df['target'].astype(str)
# val_df['target'] = val_df['target'].astype(str)
# test_df['target'] = test_df['target'].astype(str)

# print(f"Training samples: {len(train_df)}")
# print(f"Validation samples: {len(val_df)}")
# print(f"Test samples: {len(test_df)}")

# # Image preprocessing
# IMG_SIZE = (224, 224)
# BATCH_SIZE = 32

# train_datagen = ImageDataGenerator(
#     rescale=1.0/255.0,
#     rotation_range=20,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     horizontal_flip=True
# )

# val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# # Prepare data generators for train, validation, and test sets
# train_generator = train_datagen.flow_from_dataframe(
#     dataframe=train_df,
#     directory=data_dir,
#     x_col='image',
#     y_col=class_column,
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     class_mode='binary',  # Binary classification
# )

# val_generator = val_test_datagen.flow_from_dataframe(
#     dataframe=val_df,
#     directory=data_dir,
#     x_col='image',
#     y_col=class_column,
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     class_mode='binary',  # Binary classification
# )

# test_generator = val_test_datagen.flow_from_dataframe(
#     dataframe=test_df,
#     directory=data_dir,
#     x_col='image',
#     y_col=class_column,
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     class_mode='binary',  # Binary classification
#     shuffle=False  # No need to shuffle for evaluation
# )


# # Define model
# def build_model(input_shape):
#     inputs = Input(shape=input_shape)

#     # Encoder: ResNet50 backbone
#     resnet = ResNet50(weights='imagenet', include_top=False, input_tensor=inputs)
#     for layer in resnet.layers:  # Freeze most layers
#         layer.trainable = False    
#     x = resnet.output

#     # Upsampling decoder for segmentation-style output
#     x = Conv2D(512, (1, 1), activation='relu')(x)
#     x = Reshape((-1, 512))(x)
    
#     attention_output = MultiHeadAttention(num_heads=8, key_dim=64)(x, x)
#     x = LayerNormalization()(attention_output + x)
    
#     # Reshape the tensor back to a spatial dimension for the upsampling process
#     x = Reshape((7, 7, 512))(x)
#     x = UpSampling2D((2, 2))(x)
#     x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)

#     x = UpSampling2D((2, 2))(x)
#     x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)

#     x = UpSampling2D((2, 2))(x)
#     x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)

#     # Flatten 
#     x = Flatten()(x)
#     outputs = Dense(1, activation='sigmoid')(x)  # Binary output

#     model = Model(inputs, outputs)
#     return model

# # Check for available GPUs
# gpus = tf.config.list_physical_devices('GPU')
# if gpus:
#     print("GPUs are available!")
# else:
#     print("No GPUs available, running on CPU.")

# # Set strategy for distributed training on GPUs
# strategy = tf.distribute.MirroredStrategy() if gpus else None


# model = build_model((IMG_SIZE[0], IMG_SIZE[1], 3))
# model.summary()

# # Compile the model
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
#     loss=tf.keras.losses.BinaryCrossentropy(),
#     metrics=[tf.keras.metrics.BinaryAccuracy()]
# )

# # Training parameters
# steps_per_epoch = len(train_df) // BATCH_SIZE
# validation_steps = len(val_df) // BATCH_SIZE

# # Train the model
# history = model.fit(
#     train_generator,
#     validation_data=val_generator,
#     epochs=20,
#     steps_per_epoch=steps_per_epoch,
#     validation_steps=validation_steps
# )

# # Evaluate the model on the test set
# loss, accuracy = model.evaluate(test_generator)
# print(f"Test Accuracy: {accuracy * 100:.2f}%")


In [2]:
# import os
# import pandas as pd
# from sklearn.model_selection import train_test_split
# import tensorflow as tf
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.applications import ResNet50
# from tensorflow.keras.layers import Conv2D, Reshape, UpSampling2D, LayerNormalization, MultiHeadAttention, Flatten, Input, Dropout, Dense
# from tensorflow.keras.models import Model
# from tensorflow.keras import optimizers, losses, metrics

# # Set paths
# data_dir = "/kaggle/input/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input"  # Path to the folder containing all images
# csv_path = "/kaggle/input/isic-2019/ISIC_2019_Training_GroundTruth.csv"  # Path to the CSV file

# # Load the dataset
# labels_df = pd.read_csv(csv_path)

# # Add extensions to image filenames
# labels_df['image'] = labels_df['image'] + '.jpg'

# # Define class columns
# class_columns = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']

# # Split the dataset
# train_df, test_val_df = train_test_split(labels_df, test_size=0.2, random_state=42)
# val_df, test_df = train_test_split(test_val_df, test_size=0.5, random_state=42)

# print(f"Training samples: {len(train_df)}")
# print(f"Validation samples: {len(val_df)}")
# print(f"Test samples: {len(test_df)}")


# print(f"Validation DataFrame shape: {val_df.shape}")
# print(f"First few rows of validation data:\n{val_df.head()}")


# # Image preprocessing
# IMG_SIZE = (224, 224)
# BATCH_SIZE = 32

# train_datagen = ImageDataGenerator(
#     rescale=1.0/255.0,
#     rotation_range=20,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     horizontal_flip=True
# )

# val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# # Prepare data generators
# train_generator = train_datagen.flow_from_dataframe(
#     dataframe=train_df,
#     directory=data_dir,
#     x_col='image',
#     y_col=class_columns,
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     class_mode='raw'  # For multi-label classification
# )

# val_generator = val_test_datagen.flow_from_dataframe(
#     dataframe=val_df,
#     directory=data_dir,
#     x_col='image',
#     y_col=class_columns,
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     class_mode='raw'
# )



# test_generator = val_test_datagen.flow_from_dataframe(
#     dataframe=test_df,
#     directory=data_dir,
#     x_col='image',
#     y_col=class_columns,
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE,
#     class_mode='raw',
#     shuffle=False
# )
# from transformers import CLIPVisionModel, CLIPProcessor
# from tensorflow.keras.layers import Dense, Input, LayerNormalization, Reshape, Conv2D, UpSampling2D, Flatten
# from tensorflow.keras.models import Model

# def build_model(input_shape):
#     # Initialize CLIP Vision Backbone
#     clip_model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
#     clip_model.trainable = False  # Freeze the CLIP backbone

#     # Input Layer
#     inputs = Input(shape=input_shape)

#     # Pass input through CLIP Vision Backbone
#     clip_features = clip_model(inputs)[0]  # Extract features
#     # Output shape from CLIP might differ, ensure it fits your layers
#     x = Reshape((7, 7, -1))(clip_features)  # Example reshaping

#     # Additional Conv Layer to adapt to the rest of the architecture
#     x = Conv2D(512, (1, 1), activation='relu')(x)
#     patches = Reshape((-1, 512))(x)

#     # Multi-Head Attention
#     attention_output = MultiHeadAttention(num_heads=8, key_dim=64)(patches, patches)
#     x = LayerNormalization()(attention_output + patches)

#     # Reshape back to image-like tensor
#     x = Reshape((7, 7, 512))(x)

#     # Decoder (Upsampling Layers)
#     x = UpSampling2D((2, 2))(x)
#     x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)

#     x = UpSampling2D((2, 2))(x)
#     x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)

#     x = UpSampling2D((2, 2))(x)
#     x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)

#     # Flatten and Dense for Output
#     x = Flatten()(x)
#     outputs = Dense(9, activation='sigmoid')(x)

#     model = Model(inputs, outputs)
#     return model

# # Check for available GPUs
# gpus = tf.config.list_physical_devices('GPU')
# if gpus:
#     print("GPUs are available!")
# else:
#     print("No GPUs available, running on CPU.")

# # Set strategy for distributed training on GPUs
# strategy = tf.distribute.MirroredStrategy() if gpus else None

# # Define and compile the model within the strategy scope
# with strategy.scope():
#     model = build_model((IMG_SIZE[0], IMG_SIZE[1], 3))

#     # Compile the model
#     model.compile(
#         optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
#         loss=tf.keras.losses.BinaryCrossentropy(),
#         metrics=[tf.keras.metrics.BinaryAccuracy()]
#     )


# BATCH_SIZE=32
# steps_per_epoch = len(train_df) // BATCH_SIZE
# validation_steps = len(val_df) // BATCH_SIZE

# # Train the model
# history = model.fit(
#     train_generator,
#     validation_data=val_generator,
#     epochs=20,
#     steps_per_epoch=steps_per_epoch,
#     validation_steps=validation_steps
# )

# # Evaluate the model
# loss, accuracy = model.evaluate(test_generator)
# print(f"Test Accuracy: {accuracy * 100:.2f}%")


# New Approach

In [3]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Conv2D, Reshape, UpSampling2D, Flatten, Input, LayerNormalization, MultiHeadAttention, Dense
from tensorflow.keras.models import Model
from transformers import CLIPProcessor, CLIPModel
import numpy as np

# Set paths
data_dir = "/kaggle/input/isic-2020-jpg-224x224-resized/train-image/image"  # Path to the folder containing all images
csv_path = "/kaggle/input/isic-2020-jpg-224x224-resized/train-metadata.csv"  # Path to the CSV file

# Load the dataset
labels_df = pd.read_csv(csv_path)

# Add extensions to image filenames (assuming image names are in 'isic_id' column)
labels_df['image'] = labels_df['isic_id'] + '.jpg'

# Define class column (binary classification)
class_column = 'target'  # Assuming the column for binary labels is 'target'

# Split the dataset into train, validation, and test sets
train_df, test_val_df = train_test_split(labels_df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(test_val_df, test_size=0.5, random_state=42)

train_df['target'] = train_df['target'].astype(str)
val_df['target'] = val_df['target'].astype(str)
test_df['target'] = test_df['target'].astype(str)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

# Image preprocessing
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Prepare data generators for train, validation, and test sets
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=data_dir,
    x_col='image',
    y_col=class_column,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',  # Binary classification
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=data_dir,
    x_col='image',
    y_col=class_column,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',  # Binary classification
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=data_dir,
    x_col='image',
    y_col=class_column,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',  # Binary classification
    shuffle=False  # No need to shuffle for evaluation
)

# CLIP setup
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Function to extract patches and compute similarity scores
def extract_relevant_patches(image_batch, text_prompts, top_k=5):
    # Preprocess images for CLIP
    inputs = clip_processor(images=image_batch, return_tensors="pt", padding=True)
    image_features = clip_model.get_image_features(**inputs).detach().numpy()

    # Preprocess text prompts for CLIP
    text_inputs = clip_processor(text=text_prompts, return_tensors="pt", padding=True)
    text_features = clip_model.get_text_features(**text_inputs).detach().numpy()

    # Compute similarity scores
    similarities = np.dot(image_features, text_features.T)

    # Extract top-k patches based on similarity
    top_k_indices = np.argsort(similarities, axis=-1)[:, -top_k:]
    top_k_patches = image_batch[:, top_k_indices]
    return top_k_patches

# Build the model
def build_model(input_shape, num_patches=224):
    inputs = Input(shape=input_shape)
    x=inputs
    # inputs = Input(shape=input_shape)


    
    # # Patch-based feature extraction
    # patches = Reshape((num_patches, -1))(inputs)

    # # Multi-Head Attention
    # attention_output = MultiHeadAttention(num_heads=8, key_dim=64)(patches, patches)
    # x = LayerNormalization()(attention_output + patches)


    
    # Feed refined patches into ResNet
    resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    for layer in resnet.layers:
        layer.trainable = False

    x = resnet(x)

    # Classification head
    x = Flatten()(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    return model

# Check for available GPUs
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPUs are available!")
else:
    print("No GPUs available, running on CPU.")

# Set strategy for distributed training on GPUs
strategy = tf.distribute.MirroredStrategy() if gpus else None

model = build_model((IMG_SIZE[0], IMG_SIZE[1], 3))
model.summary()

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[tf.keras.metrics.BinaryAccuracy()]
)

# Training parameters
steps_per_epoch = len(train_df) // BATCH_SIZE
validation_steps = len(val_df) // BATCH_SIZE

# Train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps
)

# Evaluate the model on the test set
loss, accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Training samples: 26500
Validation samples: 3313
Test samples: 3313
Found 26500 validated image filenames belonging to 2 classes.
Found 3313 validated image filenames belonging to 2 classes.
Found 3313 validated image filenames belonging to 2 classes.


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


828/828 ━━━━━━━━━━━━━━━━━━━━ 2739s 3s/step - binary_accuracy: 0.9819 - loss: 0.0956 - val_binary_accuracy: 0.9836 - val_loss: 0.0846
Epoch 2/20
  1/828 ━━━━━━━━━━━━━━━━━━━━ 37:36 3s/step - binary_accuracy: 1.0000 - loss: 0.0088

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


828/828 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - binary_accuracy: 1.0000 - loss: 0.0088 - val_binary_accuracy: 0.9412 - val_loss: 0.2804
Epoch 3/20
828/828 ━━━━━━━━━━━━━━━━━━━━ 2718s 3s/step - binary_accuracy: 0.9832 - loss: 0.0877 - val_binary_accuracy: 0.9833 - val_loss: 0.0814
Epoch 4/20
828/828 ━━━━━━━━━━━━━━━━━━━━ 34s 37ms/step - binary_accuracy: 1.0000 - loss: 0.0175 - val_binary_accuracy: 1.0000 - val_loss: 0.0152
Epoch 5/20
828/828 ━━━━━━━━━━━━━━━━━━━━ 2741s 3s/step - binary_accuracy: 0.9831 - loss: 0.0883 - val_binary_accuracy: 0.9836 - val_loss: 0.0805
Epoch 6/20
828/828 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - binary_accuracy: 1.0000 - loss: 0.0217 - val_binary_accuracy: 0.9412 - val_loss: 0.2411
Epoch 7/20
828/828 ━━━━━━━━━━━━━━━━━━━━ 2724s 3s/step - binary_accuracy: 0.9817 - loss: 0.0914 - val_binary_accuracy: 0.9839 - val_loss: 0.0856
Epoch 8/20
828/828 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - binary_accuracy: 1.0000 - loss: 0.0051 - val_binary_accuracy: 0.8824 - val_loss: 0.6366
Epoch 9/2